In [ ]:
import os
import sys
import pandas as pd
import joblib
from IPython.display import display

# 1. Capiamo dove si trova il notebook (es. dentro la cartella 'notebooks')
cartella_corrente = os.getcwd()

# 2. Facciamo UN PASSO INDIETRO per trovare la cartella principale del progetto
cartella_principale = os.path.dirname(cartella_corrente)

# 3. Aggiungiamo la cartella principale al radar di Python
if cartella_principale not in sys.path:
    sys.path.append(cartella_principale)

from utils.preprocessing import build_preprocessor

# Per i percorsi dei file, partiamo sempre dalla cartella principale:
dataset_path = os.path.join(cartella_principale, 'resources', 'adapted_dataset.csv')
model_path = os.path.join(cartella_principale, 'resources', 'kmeans_model.pkl')

print(f"Caricamento dati in corso da:\n{dataset_path} ...")
df_utenti = pd.read_csv(dataset_path)

print(f"Caricamento modello in corso da:\n{model_path} ...")
pipeline = joblib.load(model_path)

# 3. Assegna il cluster a ogni utente
df_utenti['cluster'] = pipeline.predict(df_utenti)

# 4. Definisci le colonne PER LA STAMPA
numeric_features_display = [
    'career_ambition', 'openness', 'extraversion', 
    'agreeableness', 'conscientiousness', 'chronotype', 
    'spontaneity', 'emotional_intelligence'
]
categorical_features_display = ['age', 'career_field', 'communication_style', 'education']

# 5. Profilazione Numerica (Solo tratti continui)
print("\n" + "="*50)
print(" 1. MEDIE DELLE FEATURE NUMERICHE ")
print("="*50)
medie = df_utenti.groupby('cluster')[numeric_features_display].mean().round(2)
display(medie)

# --- DIZIONARIO PER LA TRADUZIONE DELL'ISTRUZIONE ---
mappa_istruzione = {
    1: "Diploma Superiore (High School)",
    2: "Diploma Universitario (Associate's)",
    3: "Laurea Triennale (Bachelor's)",
    4: "Laurea Magistrale (Master's)",
    5: "Dottorato di Ricerca (Doctorate)"
}
# ----------------------------------------------------

# 6. Profilazione Categorica e Ordinale (La Moda)
print("\n" + "="*50)
print(" 2. TRATTI CATEGORICI E ISTRUZIONE (Moda)")
print("="*50)
for cluster_id in sorted(df_utenti['cluster'].unique()):
    subset = df_utenti[df_utenti['cluster'] == cluster_id]
    print(f"\n[ CLUSTER {cluster_id} ]")
    
    for cat_col in categorical_features_display:
        valore_frequente = subset[cat_col].mode()[0]
        
        # Sostituiamo il numero freddo con la stringa tradotta
        if cat_col == 'education':
            # Prende la traduzione dal dizionario, o lascia il numero se c'è un errore nei dati
            etichetta = mappa_istruzione.get(int(valore_frequente), f"Livello Sconosciuto ({valore_frequente})")
            print(f"  - {cat_col}: {etichetta}")
        else:
            print(f"  - {cat_col}: {valore_frequente}")

# 7. SALVATAGGIO DEL DATASET CON I CLUSTER
# output_path = os.path.join(cartella_principale, 'resources', 'clustered_dataset.csv')
# df_utenti.to_csv(output_path, index=False)
# print(f"\nOperazione conclusa! Il dataset con i cluster assegnati è stato salvato in:\n{output_path}")            

Caricamento dati in corso da:
c:\Users\checc\Social_Matcher\k-means\resources\adapted_dataset.csv ...
Caricamento modello in corso da:
c:\Users\checc\Social_Matcher\k-means\resources\kmeans_model.pkl ...

 1. MEDIE DELLE FEATURE NUMERICHE 


,career_ambition,openness,extraversion,agreeableness,conscientiousness,chronotype,spontaneity,emotional_intelligence
cluster,,,,,,,,
0,0.38,0.5,0.41,0.46,0.64,0.57,0.39,0.43
1,0.59,0.5,0.67,0.59,0.58,0.54,0.43,0.63
2,0.39,0.5,0.56,0.53,0.34,0.42,0.62,0.54
3,0.63,0.5,0.36,0.43,0.44,0.47,0.55,0.40



 2. TRATTI CATEGORICI E ISTRUZIONE (Moda)

[ CLUSTER 0 ]
  - age: 31-43
  - career_field: Law
  - communication_style: Thoughtful Gestures
  - education: Diploma Universitario (Associate's)

[ CLUSTER 1 ]
  - age: 31-43
  - career_field: Entrepreneurship
  - communication_style: Shared Experiences
  - education: Laurea Triennale (Bachelor's)

[ CLUSTER 2 ]
  - age: 31-43
  - career_field: Education
  - communication_style: Practical Reliability
  - education: Diploma Universitario (Associate's)

[ CLUSTER 3 ]
  - age: 31-43
  - career_field: Creative Arts
  - communication_style: Thoughtful Gestures
  - education: Laurea Magistrale (Master's)
